In [2]:
import os
import warnings
import zipfile

import pandas as pd

warnings.filterwarnings("ignore")

In [3]:
path = r"C:\Users\bhavy\Massachusetts Institute of Technology\Truck Parking Capstone - General\Truck Stop Finder 🚚⛽\\"
# path = r"C:\Users\samcl\Massachusetts Institute of Technology\Truck Parking Capstone - Truck Stop Finder 🚚⛽\\"

In [57]:
sensor_loc = pd.read_csv(path + r"5. Source & Refrence Files\sensor_loc.csv",
                         dtype={"station_id": 'str'})
state_map = pd.read_csv(path + r"5. Source & Refrence Files\State_mapping.csv")


In [93]:
folder = path + r"5. Source & Refrence Files\2024_traffic_data"

dfs = []

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)

    # --- Case 1: ZIP file ---
    if filename.lower().endswith(".zip"):
        print(f"Opening ZIP: {filename}")

        with zipfile.ZipFile(file_path, 'r') as z:
            for inner_name in z.namelist():
                # skip folders inside zip
                if inner_name.endswith("/"):
                    continue

                # print("  Reading inside ZIP:", inner_name)

                with z.open(inner_name) as f:
                    df = pd.read_csv(f, delimiter="|")
                    dfs.append(df)

# Combine all data
combined_traffic_df = pd.concat(dfs, ignore_index=True)

Opening ZIP: apr_2024_ccs_data.zip
Opening ZIP: aug_2024_ccs_data.zip
Opening ZIP: dec_2024_ccs_data.zip
Opening ZIP: feb_2024_ccs_data.zip
Opening ZIP: jan_2024_ccs_data.zip
Opening ZIP: jul_2024_ccs_data.zip
Opening ZIP: jun_2024_ccs_data.zip
Opening ZIP: mar_2024_ccs_data.zip
Opening ZIP: may_2024_ccs_data.zip
Opening ZIP: nov_2024_ccs_data.zip
Opening ZIP: oct_2024_ccs_data.zip
Opening ZIP: sep_2024_ccs_data.zip


In [94]:
columns = ['record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
           'travel_lane', 'year_record', 'month_record', 'day_record',
           'day_of_week', 'hour_00', 'hour_01', 'hour_02', 'hour_03', 'hour_04',
           'hour_05', 'hour_06', 'hour_07', 'hour_08', 'hour_09', 'hour_10',
           'hour_11', 'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16',
           'hour_17', 'hour_18', 'hour_19', 'hour_20', 'hour_21', 'hour_22',
           'hour_23', 'restrictions', 'State']

In [95]:
print(combined_traffic_df.shape)
combined_traffic_df = pd.merge(combined_traffic_df, state_map, on="state_code", how="left")
print(combined_traffic_df.shape)
combined_traffic_df["station_id"] = combined_traffic_df["station_id"].astype(str)

(7814424, 35)
(7814424, 44)


In [96]:
traffic_df = pd.DataFrame()

In [97]:
combined_traffic_df_merge = pd.merge(combined_traffic_df, sensor_loc, left_on=["station_id", "State"],
                                     right_on=["Station Id", "State"], how="left")
traffic_df = pd.concat([traffic_df, combined_traffic_df_merge[~combined_traffic_df_merge["Latitude"].isnull()]],
                       ignore_index=True)
combined_traffic_df = combined_traffic_df_merge[combined_traffic_df_merge["Latitude"].isnull()][columns].copy()
print(f"Unmatched records are: {len(combined_traffic_df)}")

Unmatched records are: 809019


In [98]:
combined_traffic_df["station_id"] = combined_traffic_df["station_id"].str.lstrip("0")

In [99]:
combined_traffic_df_merge = pd.merge(combined_traffic_df, sensor_loc, left_on=["station_id", "State"],
                                     right_on=["Station Id", "State"], how="left")
traffic_df = pd.concat([traffic_df, combined_traffic_df_merge[~combined_traffic_df_merge["Latitude"].isnull()]],
                       ignore_index=True)
combined_traffic_df = combined_traffic_df_merge[combined_traffic_df_merge["Latitude"].isnull()][columns].copy()
print(f"Unmatched records are: {len(combined_traffic_df)}")

Unmatched records are: 668


In [86]:
combined_traffic_df["station_id"].unique()

array(['E131'], dtype=object)

In [103]:
traffic_df.columns

Index(['record_type', 'state_code', 'f_system', 'station_id', 'travel_dir',
       'travel_lane', 'year_record', 'month_record', 'day_record',
       'day_of_week', 'hour_00', 'hour_01', 'hour_02', 'hour_03', 'hour_04',
       'hour_05', 'hour_06', 'hour_07', 'hour_08', 'hour_09', 'hour_10',
       'hour_11', 'hour_12', 'hour_13', 'hour_14', 'hour_15', 'hour_16',
       'hour_17', 'hour_18', 'hour_19', 'hour_20', 'hour_21', 'hour_22',
       'hour_23', 'restrictions', 'State', 'Unnamed: 2', 'Unnamed: 3',
       'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8',
       'Unnamed: 9', 'Latitude', 'Longitude', 'Functional Class',
       'Station Id'],
      dtype='object')

In [102]:
traffic_df

,record_type,state_code,f_system,station_id,travel_dir,travel_lane,year_record,month_record,day_record,day_of_week,...,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Latitude,Longitude,Functional Class,Station Id
0,V,2,1R,101,1,1,2024,4,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,62.35135,-150.25191,1R,101
1,V,2,1R,101,1,1,2024,4,2,3,...,NaN,NaN,NaN,NaN,NaN,NaN,62.35135,-150.25191,1R,101
2,V,2,1R,101,1,1,2024,4,3,4,...,NaN,NaN,NaN,NaN,NaN,NaN,62.35135,-150.25191,1R,101
3,V,2,1R,101,1,1,2024,4,4,5,...,NaN,NaN,NaN,NaN,NaN,NaN,62.35135,-150.25191,1R,101
4,V,2,1R,101,1,1,2024,4,5,6,...,NaN,NaN,NaN,NaN,NaN,NaN,62.35135,-150.25191,1R,101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7813751,V,56,3U,202,5,2,2024,9,11,4,...,NaN,NaN,NaN,NaN,NaN,NaN,41.15362,-104.78540,3U,202
7813752,V,56,3U,202,5,2,2024,9,12,5,...,NaN,NaN,NaN,NaN,NaN,NaN,41.15362,-104.78540,3U,202
7813753,V,56,3U,202,5,2,2024,9,13,6,...,NaN,NaN,NaN,NaN,NaN,NaN,41.15362,-104.78540,3U,202
7813754,V,56,3U,202,5,2,2024,9,14,7,...,NaN,NaN,NaN,NaN,NaN,NaN,41.15362,-104.78540,3U,202
